In [3]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path

In [4]:
CLASS_MAP = {
    0: "Inclusion",
    1: "Patches",
    2: "Scratches"
}

PRETTY_SPLIT = {
    "val": "Validation",
    "test": "Test",
    "train": "Train"
}

In [5]:
def test_summary(data_dir: Path):
    """Return summary of client data including splits and class distributions"""
    rows = []
    file_name = data_dir.name
    label_root = data_dir / "labels" 
    images = set(os.listdir(data_dir / "images"))
    labels = set(os.listdir(data_dir / "labels"))

    for image in images:
        image = Path(image)
        label_file = image.with_suffix(".txt").name

        # Ensures that image has label
        assert label_file in labels
        
        # Get label information
        content = (label_root / label_file).read_text(encoding="utf-8")
        class_counts = [0, 0, 0]
        for row in content.split("\n"):
            if row == "":
                continue
            class_id = int(row[0][0])
            class_counts[class_id] += 1
        
        entry = [file_name, "test", Path(label_file).stem] + class_counts
        rows.append(entry)
    return pd.DataFrame(rows, columns=["client", "split", "file_name"] + list(CLASS_MAP.values()))

def client_summary(data_dir: Path):
    """Return summary of test data class distributions"""
    rows = []
    file_name = data_dir.name

    for data_folder in ["train", "val"]:
        label_root = data_dir / "labels" / data_folder
        images = set(os.listdir(data_dir / "images" / data_folder))
        labels = set(os.listdir(data_dir / "labels" / data_folder))

        for image in images:
            image = Path(image)
            label_file = image.with_suffix(".txt").name

            # Ensures that image has label
            assert label_file in labels
            
            # Get label information
            content = (label_root / label_file).read_text(encoding="utf-8")
            class_counts = [0, 0, 0]
            for row in content.split("\n"):
                if row == "":
                    continue
                class_id = int(row[0][0])
                class_counts[class_id] += 1
            
            entry = [file_name, data_folder, Path(label_file).stem] + class_counts
            rows.append(entry)
    return pd.DataFrame(rows, columns=["client", "split", "file_name"] + list(CLASS_MAP.values()))

In [8]:
def _split_report(df, split):
    split_df = df[df["split"] == split]
    class_0 = split_df[split_df[CLASS_MAP[0]] > 0]
    class_1 = split_df[split_df[CLASS_MAP[1]] > 0]
    class_2 = split_df[split_df[CLASS_MAP[2]] > 0]

    class_columns = [CLASS_MAP[0], CLASS_MAP[1], CLASS_MAP[2]]
    non_zero_counts = (split_df[class_columns] > 0).sum(axis=1)
    empty = split_df[non_zero_counts == 0]
    multiple = split_df[non_zero_counts >= 2]

  
    print(f"{split} contains {len(split_df)} images.")
    print(f"    - {len(class_0)} images containing {CLASS_MAP[0]}")
    print(f"    - {len(class_1)} images containing {CLASS_MAP[1]}")
    print(f"    - {len(class_2)} images containing {CLASS_MAP[2]}")
    print(f"    - {len(empty)} images containing any labels")
    print(f"    - {len(multiple)} images containing multi-class labels")

    print(f"Total Instance Summary:")
    for i in range(3):
        defect_name = CLASS_MAP[i]
        print(f"    - {defect_name}: {split_df[defect_name].sum()}")

def main(data_root: Path):
    clients = ["client_0", "client_1", "client_2"]
    for client in clients:
        df = client_summary(data_root / client)

        print("=" * 50)
        print(f"Client: {client}")
        
        for split in ["train","val"]:
            print("\n" + PRETTY_SPLIT[split])
            _split_report(df, split)
    
    print("=" * 50)
    print(PRETTY_SPLIT["test"])
    df = test_summary(data_root / "test")
    _split_report(df, "test", )
    
    

In [9]:
main(Path("../neu_data"))

Client: client_0

Train
train contains 457 images.
    - 234 images containing Inclusion
    - 143 images containing Patches
    - 112 images containing Scratches
    - 0 images containing any labels
    - 32 images containing multi-class labels
Total Instance Summary:
    - Inclusion: 641
    - Patches: 390
    - Scratches: 202

Validation
val contains 30 images.
    - 12 images containing Inclusion
    - 10 images containing Patches
    - 10 images containing Scratches
    - 0 images containing any labels
    - 2 images containing multi-class labels
Total Instance Summary:
    - Inclusion: 23
    - Patches: 25
    - Scratches: 15
Client: client_1

Train
train contains 228 images.
    - 69 images containing Inclusion
    - 105 images containing Patches
    - 78 images containing Scratches
    - 0 images containing any labels
    - 24 images containing multi-class labels
Total Instance Summary:
    - Inclusion: 179
    - Patches: 294
    - Scratches: 151

Validation
val contains 30 ima